In [0]:
%sql
-- ==========================================================
-- 1. SEX REFERENCE CONFLICT
-- ==========================================================

SELECT
    UPPER(TRIM(raw_sex)) AS normalized_key,
    COUNT(DISTINCT UPPER(TRIM(standard_sex))) AS value_count
FROM clinical_trial_intelligence.silver.ref_sex
WHERE raw_sex IS NOT NULL
GROUP BY UPPER(TRIM(raw_sex))
HAVING COUNT(DISTINCT UPPER(TRIM(standard_sex))) > 1;


-- ==========================================================
-- 2. DIAGNOSIS REFERENCE CONFLICT
-- ==========================================================

SELECT
    UPPER(TRIM(diagnosis_code)) AS normalized_key,
    COUNT(DISTINCT TRIM(diagnosis_description)) AS value_count
FROM clinical_trial_intelligence.silver.ref_diagnosis
WHERE diagnosis_code IS NOT NULL
GROUP BY UPPER(TRIM(diagnosis_code))
HAVING COUNT(DISTINCT TRIM(diagnosis_description)) > 1;


-- ==========================================================
-- 3. SITE → STUDY CONFLICT
-- ==========================================================

SELECT
    UPPER(TRIM(site_id)) AS normalized_site_id,
    COUNT(DISTINCT UPPER(TRIM(study_id))) AS study_count
FROM clinical_trial_intelligence.silver.dim_site
WHERE site_id IS NOT NULL
GROUP BY UPPER(TRIM(site_id))
HAVING COUNT(DISTINCT UPPER(TRIM(study_id))) > 1;


-- ==========================================================
-- 4. LAB TEST REFERENCE CONFLICT
-- ==========================================================

SELECT
    UPPER(TRIM(lab_test_code)) AS normalized_lab_test_code,
    COUNT(
        DISTINCT STRUCT(
            TRIM(lab_test_name),
            TRIM(standard_unit),
            reference_low,
            reference_high
        )
    ) AS definition_count
FROM clinical_trial_intelligence.silver.ref_lab_test
WHERE lab_test_code IS NOT NULL
GROUP BY UPPER(TRIM(lab_test_code))
HAVING COUNT(
    DISTINCT STRUCT(
        TRIM(lab_test_name),
        TRIM(standard_unit),
        reference_low,
        reference_high
    )
) > 1;


-- ==========================================================
-- 5. UNIT MAPPING CONFLICT
-- ==========================================================

SELECT
    UPPER(TRIM(lab_test_code)) AS normalized_lab_test_code,
    UPPER(TRIM(raw_unit)) AS normalized_raw_unit,
    COUNT(
        DISTINCT STRUCT(
            TRIM(standard_unit),
            conversion_factor
        )
    ) AS mapping_count
FROM clinical_trial_intelligence.silver.ref_unit
WHERE lab_test_code IS NOT NULL
  AND raw_unit IS NOT NULL
GROUP BY
    UPPER(TRIM(lab_test_code)),
    UPPER(TRIM(raw_unit))
HAVING COUNT(
    DISTINCT STRUCT(
        TRIM(standard_unit),
        conversion_factor
    )
) > 1;


-- ==========================================================
-- 6. SEVERITY REFERENCE CONFLICT
-- ==========================================================

SELECT
    UPPER(TRIM(raw_severity)) AS normalized_key,
    COUNT(
        DISTINCT STRUCT(
            UPPER(TRIM(standard_severity)),
            severity_rank
        )
    ) AS mapping_count
FROM clinical_trial_intelligence.silver.ref_severity
WHERE raw_severity IS NOT NULL
GROUP BY UPPER(TRIM(raw_severity))
HAVING COUNT(
    DISTINCT STRUCT(
        UPPER(TRIM(standard_severity)),
        severity_rank
    )
) > 1;

In [0]:
%sql
-- ============================================================
-- SILVER / QUARANTINE RECONCILIATION
-- ============================================================

SELECT
    'visits' AS entity,

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.bronze.edc_visits)
        AS bronze_rows,

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.silver.visits)
        AS silver_rows,

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.quarantine.visits)
        AS quarantine_rows

UNION ALL

SELECT
    'lab_results',

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.bronze.lab_results),

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.silver.lab_results),

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.quarantine.lab_results)

UNION ALL

SELECT
    'adverse_events',

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.bronze.safety_adverse_events),

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.silver.adverse_events),

    (SELECT COUNT(*)
     FROM clinical_trial_intelligence.quarantine.adverse_events);

In [0]:
%sql
-- ============================================================
-- 1. VISITS - QUARANTINE REASONS
-- ============================================================

SELECT
    _dq_failures,
    COUNT(*) AS row_count
FROM clinical_trial_intelligence.quarantine.visits
GROUP BY _dq_failures
ORDER BY row_count DESC;

In [0]:
%sql
-- ============================================================
-- 2. LAB RESULTS - QUARANTINE REASONS
-- ============================================================

SELECT
    _dq_failures,
    COUNT(*) AS row_count
FROM clinical_trial_intelligence.quarantine.lab_results
GROUP BY _dq_failures
ORDER BY row_count DESC;

In [0]:
%sql
-- ============================================================
-- 3. ADVERSE EVENTS - QUARANTINE REASONS
-- ============================================================

SELECT
    _dq_failures,
    COUNT(*) AS row_count
FROM clinical_trial_intelligence.quarantine.adverse_events
GROUP BY _dq_failures
ORDER BY row_count DESC;

In [0]:
%sql
-- ============================================================
-- DIAGNOSE VISIT ↔ SUBJECT RELATIONSHIP
-- ============================================================

WITH current_subjects AS (
    SELECT
        UPPER(TRIM(subject_id)) AS subject_id,
        UPPER(TRIM(study_id))   AS subject_study_id,
        UPPER(TRIM(site_id))    AS subject_site_id
    FROM clinical_trial_intelligence.silver.subjects
    WHERE __END_AT IS NULL
)

SELECT
    COUNT(*) AS total_visits,

    SUM(
        CASE
            WHEN s.subject_id IS NULL THEN 1
            ELSE 0
        END
    ) AS subject_not_found,

    SUM(
        CASE
            WHEN s.subject_id IS NOT NULL
             AND UPPER(TRIM(v.study_id)) <> s.subject_study_id
            THEN 1
            ELSE 0
        END
    ) AS study_mismatch,

    SUM(
        CASE
            WHEN s.subject_id IS NOT NULL
             AND UPPER(TRIM(v.site_id)) <> s.subject_site_id
            THEN 1
            ELSE 0
        END
    ) AS site_mismatch,

    SUM(
        CASE
            WHEN s.subject_id IS NOT NULL
             AND UPPER(TRIM(v.study_id)) = s.subject_study_id
             AND UPPER(TRIM(v.site_id)) = s.subject_site_id
            THEN 1
            ELSE 0
        END
    ) AS valid_relationship

FROM clinical_trial_intelligence.bronze.edc_visits v

LEFT JOIN current_subjects s
    ON UPPER(TRIM(v.subject_id)) = s.subject_id;

In [0]:
%sql
SELECT
    onset_date,
    COUNT(*) AS row_count
FROM clinical_trial_intelligence.bronze.safety_adverse_events
GROUP BY onset_date
ORDER BY row_count DESC
LIMIT 30;

In [0]:
%sql
SELECT
    MIN(onset_date) AS min_raw_onset_date,
    MAX(onset_date) AS max_raw_onset_date
FROM clinical_trial_intelligence.bronze.safety_adverse_events;

In [0]:
%sql

SELECT 'subjects' AS entity, COUNT(*) AS silver_rows
FROM clinical_trial_intelligence.silver.subjects

UNION ALL

SELECT 'visits', COUNT(*)
FROM clinical_trial_intelligence.silver.visits

UNION ALL

SELECT 'lab_results', COUNT(*)
FROM clinical_trial_intelligence.silver.lab_results

UNION ALL

SELECT 'adverse_events', COUNT(*)
FROM clinical_trial_intelligence.silver.adverse_events;

In [0]:
%sql

SELECT 'subjects' AS entity, COUNT(*) AS silver_rows
FROM clinical_trial_intelligence.silver.subjects

UNION ALL

SELECT 'visits', COUNT(*)
FROM clinical_trial_intelligence.silver.visits

UNION ALL

SELECT 'lab_results', COUNT(*)
FROM clinical_trial_intelligence.silver.lab_results

UNION ALL

SELECT 'adverse_events', COUNT(*)
FROM clinical_trial_intelligence.silver.adverse_events;